In [29]:
import pandas as pd

df = pd.read_csv("../data/processed/repositories.csv", keep_default_na=False)

In [30]:
df.head(1)

,Full Name,Repository Name,Description,Topics,Primary Language,Stars Count,Forks Count,Updated At,Domain,combined_text
0,robertpeteuil/multi-cloud-control,multi-cloud-control,Multi cloud control of VM Instances across AWS...,"alibaba-cloud, alibaba-cloud-cli, alibabacloud...",Python,35,10,2024-03-14 22:40:50+00:00,Cloud Computing,multi cloud control of vm instances across aws...


**Vectorizer**

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1,2),
    min_df= 2
)

X = vectorizer.fit_transform(df['combined_text'])

In [32]:
X.shape

(4253, 10000)

In [33]:
vectorizer.get_feature_names_out()[:50]

array(['10', '10 windows', '100', '100 days', '100 free', '100 languages',
       '1000', '100daysofcode', '1011', '1024', '10x', '10x faster', '11',
       '118', '12', '12 weeks', '13', '150', '16', '180', '1wire', '1x',
       '20', '20 components', '2017', '2018', '2019', '2020', '2020 3d',
       '2021', '2022', '2022 ai', '2023', '2024', '2024 agent',
       '2024 computervision', '2025', '2025 ai', '2026', '2026 awesome',
       '2026 codinginterviewquestions', '21', '23', '24', '24 lessons',
       '24ghz', '25', '26', '2d', '2d 2dframework'], dtype=object)

In [34]:
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
similarities = cosine_similarity(X[0], X)

In [36]:
similarities.shape

(1, 4253)

In [37]:
similarities[0][:10]

array([1.        , 0.06922728, 0.15419042, 0.        , 0.11194199,
       0.21037124, 0.07860243, 0.09429201, 0.11954501, 0.09140685])

In [38]:
similarity_score = list(enumerate(similarities[0]))

In [39]:
similarity_score = sorted(
    similarity_score,
    key=lambda x: x[1],
    reverse=True
)

In [40]:
similarity_score[:10]

[(0, np.float64(1.0)),
 (39, np.float64(0.45839454644555594)),
 (867, np.float64(0.40230539279047683)),
 (14, np.float64(0.3901551360865684)),
 (3518, np.float64(0.3886444712717679)),
 (320, np.float64(0.37707244924321653)),
 (123, np.float64(0.317334282608472)),
 (1694, np.float64(0.28492855693763414)),
 (2054, np.float64(0.276503329695475)),
 (126, np.float64(0.2763770083565774))]

In [41]:
indices = [i[0] for i in similarity_score[1:11]]

df.iloc[indices][["Repository Name", "Description", "Primary Language", "Stars Count"]]

,Repository Name,Description,Primary Language,Stars Count
39,alibabacloud-console-design,阿里云管平台研发解决方案,TypeScript,83
867,Cloud-Product-Mapping,"All major services between AWS, Azure, and GCP...",,907
14,cloud-cheat-sheets,My handmade cheat-sheets for different AWS ser...,,97
3518,Cloud-Free-Tier-Comparison,Comparing the free tier offers of the major cl...,,6862
320,skyplane,🔥 Blazing fast bulk data transfers between any...,Python,1211
123,terraform-provider-iterative,☁️ Terraform plugin for machine learning workl...,Go,295
1694,docker-android,Android in docker solution with noVNC supporte...,Python,14471
2054,cb-tumblebug,Cloud-Barista Multi-Cloud Infra Management Fra...,Go,80
126,AzureR,Family of packages for interacting with Azure ...,,197
81,90DaysOfGoogleCloudPlatform,,,83


**Recommendation Function**

In [15]:
# Recommendation Function

def recommend(repo_name, n=10):

    repo_indices = df.index[df["Repository Name"] == repo_name].tolist()

    if not repo_indices:
        return "Repository not found"

    idx = repo_indices[0]

    repo_vector = X[idx]

    similarities = cosine_similarity(repo_vector, X)[0] # 1-d array

    similarity_scores = list(enumerate(similarities))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the repo itself
    similarity_scores = similarity_scores[1:n+1]

    indices = [i[0] for i in similarity_scores]
    scores = [i[1] for i in similarity_scores]

    recommendations = df.iloc[indices][[
        "Repository Name",
        "Description",
        "Domain",
        "Primary Language",
        "Stars Count",
        "Forks Count",
        "Updated At"
    ]].copy()

    recommendations["Similarity"] = scores

    return recommendations


In [134]:
recommend('BitNet', 10)

,Repository Name,Description,Domain,Primary Language,Stars Count,Forks Count,Updated At,Similarity
416,awesome-llm-apps,Collection of awesome LLM apps with AI Agents ...,Python,Python,104972,15309,2026-04-10T10:09:54Z,0.233843
494,crewAI,"Framework for orchestrating role-playing, auto...",Python,Python,48506,6618,2026-04-10T10:13:51Z,0.227672
1156,transmission,Official Transmission BitTorrent client reposi...,C++,C++,14546,1367,2026-04-10T02:58:19Z,0.187202
1001,llama.cpp,LLM inference in C/C++,C++,C++,102889,16637,2026-04-10T10:18:15Z,0.182403
463,llama,Inference code for Llama models,Python,Python,59313,9833,2026-04-10T07:40:00Z,0.181132
436,vllm,A high-throughput and memory-efficient inferen...,Python,Python,75996,15412,2026-04-10T10:15:32Z,0.181025
417,DeepSeek-V3,,Python,Python,102543,16629,2026-04-10T09:43:01Z,0.179693
499,MediaCrawler,小红书笔记 | 评论爬虫、抖音视频 | 评论爬虫、快手视频 | 评论爬虫、B 站视频 ｜ 评...,Python,Python,47612,10236,2026-04-10T10:14:26Z,0.179693
567,jieba,结巴中文分词,Python,Python,34842,6705,2026-04-10T07:39:36Z,0.179693
570,TaskMatrix,,Python,Python,34173,3240,2026-04-10T07:44:12Z,0.179693


**EVALUATION**

In [27]:
eval_repos = df["Repository Name"].sample(20, random_state=42 ).tolist()

evaluation = []

for repo in eval_repos:
    recommendations = recommend(repo, 5)

    for _, row in recommendations.iterrows():
        evaluation.append({
            "Query Repo": repo,
            "Recommended Repo": row["Repository Name"],
            "Similarity": row["Similarity"],
            "Relevant": None
        })

eval_df = pd.DataFrame(evaluation)

In [28]:
eval_df

,Query Repo,Recommended Repo,Similarity,Relevant
0,nautilus_trader,nautilus_trader,0.984953,None
1,nautilus_trader,nautilus_trader,0.973761,None
2,nautilus_trader,TradingAgents,0.208087,None
3,nautilus_trader,NextTrade,0.186262,None
4,nautilus_trader,pybroker,0.166330,None
...,...,...,...,...
95,meilisearch,qdrant,0.383548,None
96,meilisearch,qdrant,0.369514,None
97,meilisearch,typesense,0.345948,None
98,meilisearch,OpenSearch,0.344509,None


## EXPERIMENTS ##

**Checking Domain feature's influence**

In [138]:
df["text_without_domain"] = (
    df["Description"] + " " +
    df["Topics"]
)

df["text_with_domain"] = (
    df["Description"] + " " +
    df["Topics"] + " " +
    df["Domain"]
)

In [139]:
vectorizer_a = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_without_domain = vectorizer_a.fit_transform(
    df["text_without_domain"]
)

In [140]:
vectorizer_b = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_with_domain = vectorizer_b.fit_transform(
    df["text_with_domain"]
)

In [141]:
X_without_domain.shape


(4997, 10000)

In [142]:
X_with_domain.shape

(4997, 10000)

In [143]:
from sklearn.metrics.pairwise import cosine_similarity

bitnet_idx = df.index[
    df["Repository Name"] == "BitNet"
][0]

sim_without = cosine_similarity(
    X_without_domain[bitnet_idx],
    X_without_domain
)[0]

sim_with = cosine_similarity(
    X_with_domain[bitnet_idx],
    X_with_domain
)[0]

In [144]:
top_without = sim_without.argsort()[::-1][1:6]
top_with = sim_with.argsort()[::-1][1:6]

In [145]:
print("WITHOUT DOMAIN")
print(df.iloc[top_without][["Repository Name", "Domain"]])

print("\nWITH DOMAIN")
print(df.iloc[top_with][["Repository Name", "Domain"]])

WITHOUT DOMAIN
          Repository Name         Domain
2611  Reverse-Engineering  Cybersecurity
1156         transmission            C++
1001            llama.cpp            C++
463                 llama         Python
4695      agibot_x1_infer       Robotics

WITH DOMAIN
          Repository Name         Domain
416      awesome-llm-apps         Python
494                crewAI         Python
1156         transmission            C++
2611  Reverse-Engineering  Cybersecurity
463                 llama         Python


*Domain provides useful categorical information, but it shouldn't be the main source of similarity.*